# Lab 02: Harmonic Oscillator — Exercise Solutions

Reference solutions for the exercises in [Lesson 02](lesson-02.ipynb).

Try each exercise yourself before reading the solutions!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Make our lesson modules importable
sys.path.insert(0, '.')                           # harmonic.py (this lesson)
sys.path.insert(0, '../lesson-01-particle-in-a-box')  # quantum1d.py (Lesson 01 solver)

from harmonic import HarmonicOscillator, HARTREE_TO_EV, HARTREE_TO_WAVENUMBER
from quantum1d import QuantumSystem1D

---
## Exercise 1: Zero-Point Energy from the Uncertainty Principle

Compute $\Delta x$ and $\Delta p$ for the ground state and verify
$\Delta x \cdot \Delta p = \frac{1}{2}$.

In [ ]:
omega = 1.0
ho = HarmonicOscillator(omega)

# ====== Analytical calculation ======
# For the ground state φ_0 = (ω/π)^{1/4} exp(-ωx²/2), which is a Gaussian:
#
# <x> = 0              (symmetric about x=0, so the mean is zero)
# <x²> = 1/(2ω)        (standard result for a Gaussian with width 1/√ω)
# Δx = √(<x²> - <x>²) = √(1/(2ω)) = 1/√(2ω)
#
# <p> = 0              (symmetric state → no net momentum)
# <p²> = ω/2           (from the virial theorem: <T> = <V> = E₀/2 = ω/4,
#                        and <T> = <p²>/2, so <p²> = ω/2)
# Δp = √(<p²>) = √(ω/2)

Delta_x_exact = 1.0 / np.sqrt(2 * omega)
Delta_p_exact = np.sqrt(omega / 2)
product_exact = Delta_x_exact * Delta_p_exact

print('=== Analytical ===')
print(f'Δx = 1/√(2ω) = {Delta_x_exact:.6f} bohr')
print(f'Δp = √(ω/2)  = {Delta_p_exact:.6f} ℏ/bohr')
print(f'Δx·Δp = {product_exact:.6f} (expect 1/2 = 0.500000)')

# ====== Numerical calculation ======
# Set up a solver just for the ground state, with a wide domain for accuracy.
x_min, x_max = ho.suggest_domain(0, n_sigma=6.0)
system = QuantumSystem1D(x_min, x_max, 500,
                         V_func=lambda x: 0.5 * omega**2 * x**2)
system.solve(1)

x = system.x        # grid positions (interior points)
dx = system.dx       # grid spacing
phi = system.states[:, 0]  # ground-state wavefunction on the grid

# Compute <x> and <x²> as grid integrals: <A> = Σ φ(xᵢ) A(xᵢ) φ(xᵢ) dx
x_mean = np.sum(phi * x * phi) * dx         # should be ~0
x2_mean = np.sum(phi * x**2 * phi) * dx     # should be 1/(2ω)
Delta_x_num = np.sqrt(x2_mean - x_mean**2)

# Compute <p²> using the finite-difference approximation to -d²/dx²:
# <p²> = <φ|p̂²|φ> = -∫ φ(x) d²φ/dx² dx  (since p̂² = -d²/dx²)
# np.gradient computes centered differences: f'(x) ≈ (f(x+dx) - f(x-dx))/(2dx)
# Applied twice gives the second derivative d²φ/dx².
d2phi = np.gradient(np.gradient(phi, dx), dx)
p2_mean = -np.sum(phi * d2phi) * dx
Delta_p_num = np.sqrt(p2_mean)  # <p> = 0 by symmetry, so Δp = √<p²>

print(f'\n=== Numerical (N=500) ===')
print(f'Δx = {Delta_x_num:.6f} bohr')
print(f'Δp = {Delta_p_num:.6f} ℏ/bohr')
print(f'Δx·Δp = {Delta_x_num * Delta_p_num:.6f}')

# ====== Re-derivation of E₀ from the uncertainty principle ======
# The total energy is E = <p²>/2 + ω²<x²>/2 = Δp²/2 + ω²Δx²/2.
# The uncertainty principle constrains Δx·Δp ≥ ½, so Δp ≥ 1/(2Δx).
# Substituting: E(Δx) = 1/(8Δx²) + ω²Δx²/2.
# Minimizing over Δx: dE/dΔx = 0 → Δx = 1/√(2ω) → E_min = ω/2.
print(f'\n--- Re-derivation of E_0 ---')
print(f'E_0 = ω/2 = {omega/2:.4f} hartree ✓')

---
## Exercise 2: Mass Dependence — Diatomic Vibration

Compute vibrational frequency for HCl and the deuterium isotope shift.

In [ ]:
# ====== Physical constants in atomic units ======
# In atomic units, all masses are measured in electron masses (m_e).
# 1 proton = 1836.15 m_e, 1 neutron ≈ 1838.68 m_e.
m_H = 1836.15        # proton mass in m_e
m_D = 2 * 1836.15    # deuteron mass in m_e (≈ proton + neutron)
m_Cl = 35 * 1822.89  # Cl-35: 35 nucleons × average nucleon mass

# Reduced mass: μ = m₁·m₂/(m₁+m₂)
# This is the effective mass that appears in the Schrödinger equation
# when you separate center-of-mass and relative coordinates.
mu_HCl = m_H * m_Cl / (m_H + m_Cl)
mu_DCl = m_D * m_Cl / (m_D + m_Cl)

print(f'μ(HCl) = {mu_HCl:.1f} m_e')
print(f'μ(DCl) = {mu_DCl:.1f} m_e')
print(f'Ratio μ(DCl)/μ(HCl) = {mu_DCl/mu_HCl:.4f}')

# (a) ω = √(k/μ): since k is a property of the bond (same for H-Cl and D-Cl),
# replacing H with D only changes μ. Therefore ω ∝ 1/√μ.
print(f'\n(a) ω ∝ 1/√μ, so ω(DCl)/ω(HCl) = √(μ_HCl/μ_DCl) = {np.sqrt(mu_HCl/mu_DCl):.4f}')

# (b) Convert the force constant k from SI to atomic units.
# In SI: k = 480 N/m. In atomic units: k [Ha/bohr²] = k [N/m] × bohr² / hartree.
bohr_m = 5.29177e-11    # 1 bohr in meters
hartree_J = 4.35974e-18  # 1 hartree in joules

k_SI = 480  # N/m (spring constant of the HCl bond)
k_au = k_SI * bohr_m**2 / hartree_J
print(f'\n(b) k = {k_SI} N/m = {k_au:.6f} hartree/bohr²')

# ω = √(k/μ) in atomic units → convert to cm⁻¹ for spectroscopic comparison
omega_HCl = np.sqrt(k_au / mu_HCl)
freq_HCl = omega_HCl * HARTREE_TO_WAVENUMBER
print(f'ω(HCl) = {omega_HCl:.6f} hartree')
print(f'ν̃(HCl) = {freq_HCl:.0f} cm⁻¹ (experimental: ~2886 cm⁻¹)')

# (c) Isotope shift: replacing H → D doubles the mass, so ω drops by ~√2.
# This is a purely mass effect — the electronic potential is unchanged.
omega_DCl = np.sqrt(k_au / mu_DCl)
freq_DCl = omega_DCl * HARTREE_TO_WAVENUMBER
print(f'\n(c) ν̃(DCl) = {freq_DCl:.0f} cm⁻¹ (experimental: ~2091 cm⁻¹)')
print(f'Isotope shift: {freq_HCl - freq_DCl:.0f} cm⁻¹')
print(f'Ratio ν̃(DCl)/ν̃(HCl) = {freq_DCl/freq_HCl:.4f} (expect ~1/√2 = {1/np.sqrt(2):.4f})')

---
## Exercise 3: Coherent States

Build a coherent state and show it oscillates like a classical particle.

In [ ]:
# ====== Coherent state: the "most classical" quantum state ======
# A coherent state |α⟩ is a superposition of energy eigenstates whose
# probability density oscillates like a classical particle — it maintains
# its Gaussian shape and tracks the classical trajectory x(t) = A cos(ωt).

omega = 1.0
ho = HarmonicOscillator(omega)
alpha = 2.0  # real α → initial displacement, no initial momentum

# The coherent state expansion |α⟩ = e^{-|α|²/2} Σ (αⁿ/√n!) |n⟩
# has Poisson-distributed weights peaked at n ≈ |α|² = 4.
# Truncate at n_max = 20 where the weights are negligible (~10⁻⁸).
n_max = 20

x = np.linspace(-8, 8, 1000)
T = 2 * np.pi / omega  # one full classical period
times = [0, T/8, T/4, 3*T/8, T/2, 5*T/8, 3*T/4, 7*T/8]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Precompute all HO eigenfunctions φ_n(x) and energies E_n.
# This avoids recomputing them at each time step.
phi_n = np.array([ho.wavefunction(x, n) for n in range(n_max + 1)])
energies_n = np.array([ho.energy(n) for n in range(n_max + 1)])

# Coherent state expansion coefficients: c_n = e^{-|α|²/2} · αⁿ / √(n!)
from scipy.special import factorial
c_n = np.exp(-alpha**2 / 2) * alpha**np.arange(n_max + 1) / np.sqrt(
    factorial(np.arange(n_max + 1), exact=False))

for ax, t in zip(axes.flat, times):
    # Time evolution: ψ(x,t) = Σ c_n · φ_n(x) · e^{-iE_n t}
    # Each eigenstate picks up a phase factor e^{-iE_n t}; the sum
    # produces a wavepacket that moves through space.
    psi = np.zeros_like(x, dtype=complex)
    for n in range(n_max + 1):
        psi += c_n[n] * phi_n[n] * np.exp(-1j * energies_n[n] * t)

    # Probability density |ψ(x,t)|²
    prob = np.abs(psi)**2

    # Classical trajectory: x_cl(t) = ⟨α|x̂|α⟩ = α√(2/ω) cos(ωt)
    # The coherent state's center follows the classical orbit exactly.
    x_cl = alpha * np.sqrt(2 / omega) * np.cos(omega * t)

    # Plot quantum probability density
    ax.fill_between(x, prob, alpha=0.3, color='#1f77b4')
    ax.plot(x, prob, '#1f77b4', linewidth=1.5)
    # Mark the classical position with a red dashed line
    ax.axvline(x_cl, color='red', linestyle='--', alpha=0.7, label='classical')

    # Light sketch of potential in background for spatial reference
    V = 0.5 * omega**2 * x**2
    ax.plot(x, V * 0.15, 'k-', linewidth=0.5, alpha=0.2)

    ax.set_title(f't = {t/T:.3f} T', fontsize=10)
    ax.set_xlim(-8, 8)
    ax.set_ylim(0, 0.8)
    ax.set_xlabel('x (bohr)', fontsize=9)
    if t == 0:
        ax.legend(fontsize=8)

fig.suptitle(f'Coherent State |α={alpha}⟩ — The "Most Classical" Quantum State',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/coherent_state.png', dpi=150, bbox_inches='tight')
plt.show()

print('(c) Yes! The probability density maintains its Gaussian shape and')
print('    oscillates back and forth, perfectly tracking the classical position.')
print('    This is because coherent states are eigenstates of the annihilation')
print('    operator and remain coherent under time evolution.')

---
## Exercise 4: Ladder Operator Matrix Elements

Compute $\langle m|\hat{x}|n\rangle$, $\langle m|\hat{p}|n\rangle$, $\langle m|\hat{x}^2|n\rangle$ analytically
and verify numerically.

In [ ]:
# ====== Matrix elements from ladder operators — analytical formulas ======
# The key idea: express x̂ and p̂ in terms of â⁺ and â⁻, then use
# â⁻|n⟩ = √n|n-1⟩ and â⁺|n⟩ = √(n+1)|n+1⟩ to evaluate matrix elements.
# The selection rules (which ⟨m|...|n⟩ are nonzero) follow directly from
# how many â⁺/â⁻ appear in the operator.

omega = 1.0
ho = HarmonicOscillator(omega)

# (a) x̂ = (â⁺ + â⁻)/√(2ω) → connects n to n±1 only (Δn = ±1)
def x_analytical(m, n, omega):
    if m == n + 1:
        return np.sqrt((n + 1) / (2 * omega))  # raising: â⁺|n⟩ → |n+1⟩
    elif m == n - 1:
        return np.sqrt(n / (2 * omega))          # lowering: â⁻|n⟩ → |n-1⟩
    return 0.0

# (b) p̂ = i√(ω/2)(â⁺ - â⁻) → also Δn = ±1, but with factors of i
def p_analytical(m, n, omega):
    if m == n + 1:
        return 1j * np.sqrt(omega * (n + 1) / 2)
    elif m == n - 1:
        return -1j * np.sqrt(omega * n / 2)
    return 0.0

# (c) x̂² = (â⁺ + â⁻)²/(2ω) → connects n to n±2 and n (Δn = 0, ±2)
# The expansion has â⁺â⁺, â⁺â⁻, â⁻â⁺, â⁻â⁻ terms.
def x2_analytical(m, n, omega):
    if m == n + 2:      # â⁺â⁺: raise twice
        return np.sqrt((n + 1) * (n + 2)) / (2 * omega)
    elif m == n:         # â⁺â⁻ + â⁻â⁺ = 2N̂+1: diagonal
        return (2 * n + 1) / (2 * omega)
    elif m == n - 2:    # â⁻â⁻: lower twice
        return np.sqrt(n * (n - 1)) / (2 * omega)
    return 0.0

# ====== Numerical verification on a grid ======
# Compute ⟨m|x|n⟩ = ∫ φ_m(x) · x · φ_n(x) dx using quadrature.
x_min, x_max = ho.suggest_domain(6, n_sigma=5.0)
x_grid = np.linspace(x_min, x_max, 2000)
dx = x_grid[1] - x_grid[0]

n_max = 5
# Precompute all wavefunctions on the grid
phi = np.array([ho.wavefunction(x_grid, n) for n in range(n_max + 1)])

# (a) Print <m|x|n> matrix — should be tridiagonal (nonzero only on ±1 diagonals)
print('(a) <m|x|n> matrix (numerical values, · marks zero by selection rule):')
print(f'{"":>5s}', end='')
for n in range(n_max + 1):
    print(f'{n:>10d}', end='')
print()
for m in range(n_max + 1):
    print(f'{m:>3d}  ', end='')
    for n in range(n_max + 1):
        x_mn_a = x_analytical(m, n, omega)
        # Numerical integral: ⟨m|x|n⟩ = Σ φ_m(xᵢ) · xᵢ · φ_n(xᵢ) · dx
        x_mn_n = np.sum(phi[m] * x_grid * phi[n]) * dx
        if abs(x_mn_a) > 1e-10 or abs(x_mn_n) > 1e-6:
            print(f'{x_mn_n:10.4f}', end='')
        else:
            print(f'{"·":>10s}', end='')
    print()

print(f'\nSelection rule: Δn = ±1 only (tridiagonal matrix). ✓')

# (c) Print nonzero <m|x²|n> elements — diagonal + ±2 off-diagonals
print(f'\n(c) <m|x²|n> — diagonal and ±2 off-diagonal:')
for m in range(n_max + 1):
    for n in range(n_max + 1):
        x2_a = x2_analytical(m, n, omega)
        # Numerical integral: ⟨m|x²|n⟩ = Σ φ_m(xᵢ) · xᵢ² · φ_n(xᵢ) · dx
        x2_n = np.sum(phi[m] * x_grid**2 * phi[n]) * dx
        if abs(x2_a) > 1e-10:
            print(f'  <{m}|x²|{n}> = {x2_n:.6f} (exact: {x2_a:.6f})')

print(f'\nSelection rule: Δn = 0, ±2 (pentadiagonal matrix). ✓')

---
## Exercise 5: Second-Order Perturbation Theory

Compute $E_0^{(2)}$ for $H' = \lambda x^4$ and compare to the numerical solver.

In [ ]:
omega = 1.0
ho = HarmonicOscillator(omega)
lam = 0.05

# ====== Second-order perturbation theory ======
# First-order PT gives E_n^(1) = ⟨n|H'|n⟩ (diagonal matrix element).
# Second-order PT adds contributions from ALL other states:
#
#   E_n^(2) = Σ_{m≠n} |⟨m|H'|n⟩|² / (E_n - E_m)
#
# Each term is negative (since E_n - E_m < 0 for m > n), meaning the
# second-order correction always LOWERS the energy — the system "relaxes"
# by mixing in nearby states.

# (a) Which states contribute to E_0^(2)?
# H' = λx⁴. The operator x⁴ = (â⁺ + â⁻)⁴/(2ω)² connects |0⟩ to states
# that differ by up to 4 quanta: |2⟩ and |4⟩.
# (|1⟩ and |3⟩ are forbidden by parity: x⁴ is even, so it only connects
# states of the same parity. |0⟩ is even → only even m contribute.)
print('(a) x^4 connects |0⟩ to |2⟩ and |4⟩ (Δn = ±2, ±4 off-diagonal).')

# (b) Compute E_0^(2) numerically using grid integration for matrix elements
x_min, x_max = ho.suggest_domain(8, n_sigma=5.0)
x_grid = np.linspace(x_min, x_max, 3000)
dx = x_grid[1] - x_grid[0]

# Precompute wavefunctions for states 0 through 9
phi = [ho.wavefunction(x_grid, n) for n in range(10)]

E0_2 = 0.0
print(f'\n(b) Second-order correction terms:')
for m in range(1, 10):
    # Compute ⟨m|x⁴|0⟩ as a grid integral: Σ φ_m(xᵢ) · xᵢ⁴ · φ_0(xᵢ) · dx
    mat_elem = np.sum(phi[m] * x_grid**4 * phi[0]) * dx
    if abs(mat_elem) > 1e-10:
        # Each term: λ²|⟨m|x⁴|0⟩|² / (E_0 - E_m)
        # Note the denominator is negative (E_0 < E_m), so each term < 0.
        dE = lam**2 * mat_elem**2 / (ho.energy(0) - ho.energy(m))
        E0_2 += dE
        print(f'  m={m}: ⟨{m}|x⁴|0⟩ = {mat_elem:.6f}, '
              f'contribution = {dE:.8f}')

print(f'\n  E_0^(2) = {E0_2:.8f} hartree')

# (c) Compare: E_0^(0) + E_0^(1) + E_0^(2) vs numerical (exact for this grid)
E0_0 = ho.energy(0)                              # zeroth order: ω/2
E0_1 = ho.perturbation_energy_first_order(0, lam) # first order: λ⟨0|x⁴|0⟩

# Solve the full anharmonic problem numerically — this is our "exact" reference
sys_pt = QuantumSystem1D(
    x_min, x_max, 600,
    V_func=lambda x: 0.5 * omega**2 * x**2 + lam * x**4
)
E_exact, _ = sys_pt.solve(1)

print(f'\n(c) Energy comparison for n=0, λ={lam}:')
print(f'  E_0^(0)                       = {E0_0:.8f}')
print(f'  E_0^(0) + E_0^(1)             = {E0_0 + E0_1:.8f}')
print(f'  E_0^(0) + E_0^(1) + E_0^(2)   = {E0_0 + E0_1 + E0_2:.8f}')
print(f'  Numerical (exact)              = {E_exact[0]:.8f}')
print(f'\n  Error with 1st order:  {abs(E0_0 + E0_1 - E_exact[0]):.2e}')
print(f'  Error with 2nd order:  {abs(E0_0 + E0_1 + E0_2 - E_exact[0]):.2e}')
print(f'  Improvement factor:    {abs(E0_0 + E0_1 - E_exact[0]) / abs(E0_0 + E0_1 + E0_2 - E_exact[0]):.1f}x')

---
## Exercise 6: The Morse Potential — Real Molecular Vibrations

Model H₂ with a Morse potential and compare to the harmonic approximation.

In [ ]:
# ====== The Morse potential: a realistic model for diatomic vibrations ======
# V(r) = D_e (1 - e^{-a(r - r_e)})²
#
# Unlike the harmonic oscillator:
# - Has a finite well depth D_e (the molecule can dissociate)
# - Is asymmetric: steep repulsive wall at short r, flat dissociation plateau at large r
# - Energy spacings DECREASE with n (anharmonicity), eventually reaching D_e
# - Has a finite number of bound states (harmonic has infinitely many)

# H₂ parameters in atomic units
D_e = 0.1745     # well depth / dissociation energy (hartree)
r_e = 1.401      # equilibrium bond length (bohr)
mu = 918.1       # reduced mass of H₂: ½ × proton mass (in m_e)

# ====== Compute the Morse range parameter a from experimental frequency ======
# Near the minimum, the Morse potential looks harmonic: V ≈ D_e a²(r-r_e)² = ½μω²(r-r_e)²
# So ω = a√(2D_e/μ), or equivalently a = ω√(μ/(2D_e)).
# We use the experimental fundamental frequency to set a.
omega_exp = 4400 / HARTREE_TO_WAVENUMBER  # 4400 cm⁻¹ → hartree
a = omega_exp * np.sqrt(mu / (2 * D_e))

print(f'Morse parameters for H₂:')
print(f'  D_e = {D_e:.4f} hartree = {D_e * HARTREE_TO_EV:.2f} eV')
print(f'  r_e = {r_e:.3f} bohr')
print(f'  μ   = {mu:.1f} m_e')
print(f'  a   = {a:.4f} bohr⁻¹')

# Recover the harmonic frequency from the Morse parameters (self-consistency check)
omega_harm = a * np.sqrt(2 * D_e / mu)
print(f'  ω   = {omega_harm:.6f} hartree = {omega_harm * HARTREE_TO_WAVENUMBER:.0f} cm⁻¹')

# ====== (a) Numerical solution of the Morse potential ======
# Domain: start just above r=0 (repulsive wall) and extend well past r_e
# into the dissociation region where V → D_e.
r_min = max(0.3, r_e - 1.5)  # avoid r=0 singularity
r_max = r_e + 8.0             # extend into flat dissociation plateau

def V_morse(r):
    """Morse potential: V = D_e (1 - exp(-a(r - r_e)))²"""
    return D_e * (1 - np.exp(-a * (r - r_e)))**2

# Solve with the same QuantumSystem1D from Lesson 01, using the reduced mass μ.
# The 'mass' parameter enters the kinetic energy: T = -1/(2μ) d²/dr².
N = 800
system = QuantumSystem1D(r_min, r_max, N, V_func=V_morse, mass=mu)
energies, states = system.solve(n_states=15)

# Bound states have E < D_e; states above D_e are unbound (continuum)
bound_mask = energies < D_e
n_bound = np.sum(bound_mask)
E_bound = energies[bound_mask]

print(f'\n(a) Found {n_bound} bound states.')
print(f'{"n":>3s}  {"E_n (hartree)":>14s}  {"E_n (cm⁻¹)":>12s}  {"ΔE (cm⁻¹)":>12s}')
print('-' * 48)
for n in range(min(n_bound, 10)):
    E_cm = E_bound[n] * HARTREE_TO_WAVENUMBER
    # Energy spacing: ΔE = E_n - E_{n-1}. For harmonic, this is constant (= ω).
    # For Morse, it decreases — this is the spectroscopic signature of anharmonicity.
    dE_cm = (E_bound[n] - E_bound[n-1]) * HARTREE_TO_WAVENUMBER if n > 0 else 0
    print(f'{n:3d}  {E_bound[n]:14.6f}  {E_cm:12.0f}  {dE_cm:12.0f}')

# (b) Compare spacing to harmonic approximation
print(f'\n(b) Harmonic approximation: ΔE = ω = {omega_harm * HARTREE_TO_WAVENUMBER:.0f} cm⁻¹ (constant)')
print(f'    Morse spacings decrease with n (anharmonicity).')

# (c) Fundamental frequency: the 0→1 transition
fundamental_cm = (E_bound[1] - E_bound[0]) * HARTREE_TO_WAVENUMBER
print(f'\n(c) Fundamental frequency (0→1):')
print(f'    Numerical:    {fundamental_cm:.0f} cm⁻¹')
print(f'    Experimental: 4161 cm⁻¹')
print(f'    Harmonic:     {omega_harm * HARTREE_TO_WAVENUMBER:.0f} cm⁻¹')
print(f'    The Morse fundamental is lower than the harmonic value — anharmonicity')
print(f'    always reduces the effective spacing.')

In [ ]:
# ====== Visualizing Morse vs harmonic: potential, levels, and anharmonicity ======

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

r_plot = np.linspace(r_min, r_max, 1000)
V_m = V_morse(r_plot)
# Harmonic approximation: match curvature at the minimum (same ω, centered on r_e)
V_h = 0.5 * mu * omega_harm**2 * (r_plot - r_e)**2

# --- Left panel: potential energy curve with bound-state energy levels ---
ax1.plot(r_plot, V_m, 'k-', linewidth=2, label='Morse')
ax1.plot(r_plot, V_h, 'b--', linewidth=1.5, alpha=0.5, label='Harmonic')
# Dissociation limit: the energy where the molecule falls apart
ax1.axhline(D_e, color='red', linestyle=':', alpha=0.5, label=f'$D_e$ = {D_e:.4f} Ha')

# Draw horizontal lines at each bound-state energy
for n in range(min(n_bound, 8)):
    ax1.axhline(E_bound[n], color='green', linewidth=0.8, alpha=0.5)
    ax1.text(r_max - 0.5, E_bound[n], f'n={n}', fontsize=8, va='bottom')

ax1.set_xlabel('r (bohr)', fontsize=12)
ax1.set_ylabel('V(r) (hartree)', fontsize=12)
ax1.set_title('Morse Potential — H₂', fontsize=13)
ax1.set_ylim(-0.01, D_e * 1.3)
ax1.legend(fontsize=10)

# --- Right panel: energy spacings ΔE(n→n+1) vs transition number ---
# For harmonic: constant (= ω). For Morse: decreases linearly with n,
# which is the Birge-Sponer relation used to estimate dissociation energy.
spacings = np.diff(E_bound[:min(n_bound, 10)]) * HARTREE_TO_WAVENUMBER
n_transitions = np.arange(len(spacings))

ax2.plot(n_transitions, spacings, 'ro-', linewidth=1.5, markersize=6,
         label='Morse (numerical)')
ax2.axhline(omega_harm * HARTREE_TO_WAVENUMBER, color='blue',
            linestyle='--', label='Harmonic $\\omega$')

ax2.set_xlabel('Transition (n → n+1)', fontsize=12)
ax2.set_ylabel('Spacing $\\Delta E$ (cm⁻¹)', fontsize=12)
ax2.set_title('Anharmonic Energy Spacings', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/morse_potential.png', dpi=150, bbox_inches='tight')
plt.show()